In [ ]:
# IMPORTANT: Due to Windows dynamic library / runtime state collision torch must be imported before mlrun (local running)
import torch
import mlrun

# Loads vars including: AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
from dotenv import load_dotenv

load_dotenv()

mlrun.set_environment(api_path="http://localhost:30070")

project = mlrun.load_project(
    name="legalcontractextractor", context="../"
)  # project yaml must be in this directory
# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

# Run the workflow

In [ ]:
# passing this in the argument hyperparams, will loop over every combination
# hyperparameters = {'epochs': [10, 20],
#                    'lr': [10, 20]}

run_obj = project.run_function(
    function="eval-train",  # use the name in the register.ipynb file
    params={
        "train_dataset": "raw-proc-process-raw_train_data",
        "train_dataset_tag": "20260814_1551",
        "val_dataset": "raw-proc-process-raw_validation_data",
        "val_dataset_tag": "20260814_1551",
        "test_dataset": "raw-proc-process-raw_test_data",
        "test_dataset_tag": "20260814_1551",
        "prompt": "contract_extractor_prompt",
        "prompt_tag": "20260814_1626",
        ##
        "epochs": 5,
        "batch_grad_accumulation": 16,
        "learning_rate": 2e-4,  # QLoRA requires slightly higher LR
        "lora_r": 16,
        "lora_alpha": 32,
        "early_stopping_threshold": 1e-3,
    },
    local=True,  # Run the pipeline sequence locally
    watch=True,  # Print the progress to the console
    verbose=True,
)

In [ ]:
run_obj.outputs

In [ ]:
run_obj = project.run_function(
    function="eval-train",  # use the name in the register.ipynb file
    params={
        "train_dataset": "raw-proc-process-raw_train_data",
        "train_dataset_tag": "20260816_1522",
        "val_dataset": "raw-proc-process-raw_validation_data",
        "val_dataset_tag": "20260816_1522",
        "test_dataset": "raw-proc-process-raw_test_data",
        "test_dataset_tag": "20260816_1522",
        "prompt": "contract_extractor_prompt",
        "prompt_tag": "20260814_1626",
        ## Lowered batch size and LR at the same time
        "epochs": 5,
        "batch_grad_accumulation": 8,   # Since batch halved, double EVAL_STEP size
        "learning_rate": 1.25e-4,  # Lowered from 2e-4
        "lora_r": 16,
        "lora_alpha": 32,
        "early_stopping_threshold": 1e-3,
    },
    local=True,  # Run the pipeline sequence locally
    watch=True,  # Print the progress to the console
    verbose=True,
)

In [ ]:
run_obj.outputs

In [ ]:
from datasets import load_from_disk

test_data = load_from_disk(f"s3://legal-llama-data/training/20260603_1558/test")

for i in range(5):
    print(test_data[i]["inference"])